In [ ]:
import os 
from IPython.display import display, Markdown
# from okr_data_io import *
# from utils import json_to_markdown
# from evaluate_okr import calculate_rule_score, get_llm_critique, revise_okr_with_llm

# Advanced usage with utilities
from okr_utils import load_okr_from_json, OKRReporter
from okr_evaluator import OKREvaluatorService


In [ ]:
PROJECT_DIR = os.path.join(os.getcwd(), "data", "input")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
service = OKREvaluatorService()
def run_evaluation_workflow(name_of_testcase):    
    testcase = load_okr_from_json(os.path.join(PROJECT_DIR, name_of_testcase))
    rule_result, llm_result = service.evaluate_okr(testcase)
    input_filename = name_of_testcase[:-5]  # Remove .json extension
    output_filename = f"evaluate_{input_filename}.md"
    OUTPUT_FNAME = os.path.join(os.getcwd(), "data", "output", output_filename)
    report = OKRReporter.generate_evaluation_report(testcase, rule_result.__dict__, llm_result.__dict__, output_path=OUTPUT_FNAME)
# display(Markdown(report))
    return report

# 1. OKR Quality Evaluation

Principle of OKR: F.A.C.T.S acronym
- Focus 
- Alignment 
- Commitment (Internal Clarity)
- Tracking (Measurability)
- Stretching (Ambition)

In [ ]:
# "action_plan": [
#             {{"task": string, "owner": string, "cadence": string, "blockers": string}}
#         ]

## Testcase

### Good sample

In [ ]:
TEST_CASE_1 = "good_okr_1.json"
report = run_evaluation_workflow(TEST_CASE_1)
display(Markdown(report))

In [ ]:
TEST_CASE_2 = "revised_okr_1.json"
report = run_evaluation_workflow(TEST_CASE_2)
display(Markdown(report))

In [ ]:
# --- 3. Bad Example Execution ---
example_okr = {
    "objective": "Better customer experience.",
    "key_results": [
        {"text": "Make our clients feel good when they do business with us."},
        {"text": "Motivate employees to enhance customer service"}
    ],
}

In [ ]:
TEST_CASE_3 = "bad_okr.json"
report = run_evaluation_workflow(TEST_CASE_3)
display(Markdown(report))

# Risk Estimation

In [ ]:
from okr_tracker import create_tracker_from_json, load_tracker_from_json_file
import datetime

### Example: Create Tracker from JSON File

In [31]:
testcase = "good_okr_tracker.json"
tracker_from_file = load_tracker_from_json_file(
    os.path.join(PROJECT_DIR, testcase),
    start_date=datetime.date(2026, 1, 1),
    end_date=datetime.date(2026, 6, 30)
)

# Simulate some progress
tracker_from_file.update_progress(30, notes="Q1 completed")
for kr in tracker_from_file.key_results[:2]:  # Update first 2 KRs
    kr.update_progress((kr.baseline + kr.target) / 3)  # Simulate 33% progress

# Generate report
report = format_tracker_report(tracker_from_file)
print(report)

2026-01-05 15:02:10,968 - okr_tracker - INFO - Initialized OKR tracker: Win the Indy 500. (2026-01-01 to 2026-06-30)
2026-01-05 15:02:10,969 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-06-30
2026-01-05 15:02:10,970 - okr_tracker - INFO - Initialized KR tracker: Increase average lap speed
2026-01-05 15:02:10,971 - okr_tracker - INFO - Added KR to Win the Indy 500.: Increase average lap speed
2026-01-05 15:02:10,972 - okr_tracker - WARNING - Could not parse date 'Pre-race season', using default: 2026-06-30
2026-01-05 15:02:10,972 - okr_tracker - INFO - Initialized KR tracker: Test at wind tunnel
2026-01-05 15:02:10,973 - okr_tracker - INFO - Added KR to Win the Indy 500.: Test at wind tunnel
2026-01-05 15:02:10,973 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-06-30
2026-01-05 15:02:10,973 - okr_tracker - INFO - Initialized KR tracker: Reduce average pit stop time
2026-01-05 15:02:10,974 - okr_tracker - INFO - Added KR t


OKR TRACKING REPORT

Objective: Win the Indy 500.
Period: 2026-01-01 to 2026-06-30
Days Remaining: 176

PROGRESS
--------
Actual:   13.3% (13.333333333333332/100)
Expected: 2.2%
Status:   On Track

HEALTH SCORE: 100/100 - Excellent - On track with no major concerns

KEY RESULTS (5)
-----------
On Track: 5
Behind:   0
At Risk:  0

ALERTS (0)
------
No alerts - Everything looks good!



### Example: "At Risk" Status - Progress Gap > 25%

In [32]:
# Scenario: We're 50% through the timeline, but only achieved 20% progress
# This creates a 30% gap → AT RISK status

# Load OKR from file
at_risk_tracker = load_tracker_from_json_file(
    os.path.join(PROJECT_DIR, "good_okr_1.json"),
    start_date=datetime.date(2025, 10, 1),  # Started 3 months ago
    end_date=datetime.date(2026, 4, 1)      # Ends in 3 months (6 months total)
)

print(f"📅 Timeline: Started {at_risk_tracker.start_date}, Ends {at_risk_tracker.end_date}")
print(f"⏱️  Expected Progress: {at_risk_tracker.calculate_expected_progress():.1f}%")
print()

# Update KRs with low progress (only 20% average)
print("📊 Updating Key Results with low progress...")
at_risk_tracker.key_results[0].update_progress(2)   # 20% of target (2/10 if target was 10)


# Check status
print()
print("="*60)
print(f"🚨 OKR Status: {at_risk_tracker.status.value}")
print(f"📈 Actual Progress: {at_risk_tracker.get_actual_progress_percent():.1f}%")
print(f"🎯 Expected Progress: {at_risk_tracker.calculate_expected_progress():.1f}%")
print(f"⚠️  Gap: {at_risk_tracker.calculate_expected_progress() - at_risk_tracker.get_actual_progress_percent():.1f}%")
print("="*60)

# Show alerts
alerts = at_risk_tracker.get_health_alerts()
print(f"\n🔔 Health Alerts ({len(alerts)}):")
for alert in alerts:
    print(f"  [{alert.level.value}] {alert.message}")

2026-01-05 15:02:13,974 - okr_tracker - INFO - Initialized OKR tracker: Win the Indy 500. (2025-10-01 to 2026-04-01)
2026-01-05 15:02:13,975 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-04-01
2026-01-05 15:02:13,975 - okr_tracker - INFO - Initialized KR tracker: Increase average lap speed
2026-01-05 15:02:13,976 - okr_tracker - INFO - Added KR to Win the Indy 500.: Increase average lap speed
2026-01-05 15:02:13,977 - okr_tracker - WARNING - Could not parse date 'Pre-race season', using default: 2026-04-01
2026-01-05 15:02:13,978 - okr_tracker - INFO - Initialized KR tracker: Test at wind tunnel
2026-01-05 15:02:13,978 - okr_tracker - INFO - Added KR to Win the Indy 500.: Test at wind tunnel
2026-01-05 15:02:13,979 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-04-01
2026-01-05 15:02:13,979 - okr_tracker - INFO - Initialized KR tracker: Reduce average pit stop time
2026-01-05 15:02:13,980 - okr_tracker - INFO - Added KR t

📅 Timeline: Started 2025-10-01, Ends 2026-04-01
⏱️  Expected Progress: 52.7%

📊 Updating Key Results with low progress...

🚨 OKR Status: At Risk
📈 Actual Progress: 20.0%
🎯 Expected Progress: 52.7%
⚠️  Gap: 32.7%

🔔 Health Alerts (1):
  [At Risk] Progress is off by 32.7% from expected.


### Example: "Behind" Status - Progress Gap 0-25%

In [43]:
# Scenario: We're 60% through the timeline, but achieved 45% progress
# This creates a 15% gap → BEHIND status

# Load OKR from file
behind_tracker = load_tracker_from_json_file(
    os.path.join(PROJECT_DIR, "good_okr_1.json"),
    start_date=datetime.date(2025, 8, 1),   # Started 5 months ago
    end_date=datetime.date(2026, 4, 1)      # Ends in 3 months (8 months total)
)

print(f"📅 Timeline: Started {behind_tracker.start_date}, Ends {behind_tracker.end_date}")
print(f"⏱️  Expected Progress: {behind_tracker.calculate_expected_progress():.1f}%")
print()

# Update KRs with moderate progress (average ~45%)
print("📊 Updating Key Results with moderate progress...")
behind_tracker.key_results[0].update_progress(5)    # 50% of target
behind_tracker.key_results[1].update_progress(10)   # 40% of target
behind_tracker.key_results[2].update_progress(35)   # 45% of target

# Check status
print()
print("="*60)
print(f"⚠️  OKR Status: {behind_tracker.status.value}")
print(f"📈 Actual Progress: {behind_tracker.get_actual_progress_percent():.1f}%")
print(f"🎯 Expected Progress: {behind_tracker.calculate_expected_progress():.1f}%")
print(f"📉 Gap: {behind_tracker.calculate_expected_progress() - behind_tracker.get_actual_progress_percent():.1f}%")
print("="*60)

# Show alerts
alerts = behind_tracker.get_health_alerts()
print(f"\n🔔 Health Alerts ({len(alerts)}):")
for alert in alerts:
    print(f"  [{alert.level.value}] {alert.message}")

2026-01-05 15:05:20,332 - okr_tracker - INFO - Initialized OKR tracker: Win the Indy 500. (2025-08-01 to 2026-04-01)
2026-01-05 15:05:20,332 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-04-01
2026-01-05 15:05:20,333 - okr_tracker - INFO - Initialized KR tracker: Increase average lap speed
2026-01-05 15:05:20,334 - okr_tracker - INFO - Added KR to Win the Indy 500.: Increase average lap speed
2026-01-05 15:05:20,334 - okr_tracker - WARNING - Could not parse date 'Pre-race season', using default: 2026-04-01
2026-01-05 15:05:20,335 - okr_tracker - INFO - Initialized KR tracker: Test at wind tunnel
2026-01-05 15:05:20,335 - okr_tracker - INFO - Added KR to Win the Indy 500.: Test at wind tunnel
2026-01-05 15:05:20,336 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-04-01
2026-01-05 15:05:20,336 - okr_tracker - INFO - Initialized KR tracker: Reduce average pit stop time
2026-01-05 15:05:20,336 - okr_tracker - INFO - Added KR t

📅 Timeline: Started 2025-08-01, Ends 2026-04-01
⏱️  Expected Progress: 64.6%

📊 Updating Key Results with moderate progress...

⚠️  OKR Status: Behind
📈 Actual Progress: 60.0%
🎯 Expected Progress: 64.6%
📉 Gap: 4.6%

🔔 Health Alerts (1):
  [Needs Attention] Progress is 4.6% behind expected.


### Example: "Over Expectation status"

In [44]:
# Scenario: We're 50% through the timeline, and achieved 55% progress
# Actual >= Expected → ON TRACK status

# Load OKR from file
on_track_tracker = load_tracker_from_json_file(
    os.path.join(PROJECT_DIR, "good_okr_1.json"),
    start_date=datetime.date(2025, 10, 1),
    end_date=datetime.date(2026, 4, 1)
)

print(f"📅 Timeline: Started {on_track_tracker.start_date}, Ends {on_track_tracker.end_date}")
print(f"⏱️  Expected Progress: {on_track_tracker.calculate_expected_progress():.1f}%")
print()

# Update KRs with good progress (average ~55%)
print("📊 Updating Key Results with good progress...")
on_track_tracker.key_results[0].update_progress(6)   # 60% of target
on_track_tracker.key_results[1].update_progress(55)  # 55% of target
on_track_tracker.key_results[2].update_progress(50)  # 50% of target

# Check status
print()
print("="*60)
print(f"✅ OKR Status: {on_track_tracker.status.value}")
print(f"📈 Actual Progress: {on_track_tracker.get_actual_progress_percent():.1f}%")
print(f"🎯 Expected Progress: {on_track_tracker.calculate_expected_progress():.1f}%")
gap = on_track_tracker.calculate_expected_progress() - on_track_tracker.get_actual_progress_percent()
print(f"📊 Gap: {gap:.1f}% {'(ahead)' if gap < 0 else ''}")
print("="*60)

# Show alerts
alerts = on_track_tracker.get_health_alerts()
print(f"\n🔔 Health Alerts ({len(alerts)}):")
if alerts:
    for alert in alerts:
        print(f"  [{alert.level.value}] {alert.message}")
else:
    print("  ✨ No alerts - Everything looks great!")

2026-01-05 15:05:48,634 - okr_tracker - INFO - Initialized OKR tracker: Win the Indy 500. (2025-10-01 to 2026-04-01)
2026-01-05 15:05:48,635 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-04-01
2026-01-05 15:05:48,636 - okr_tracker - INFO - Initialized KR tracker: Increase average lap speed
2026-01-05 15:05:48,636 - okr_tracker - INFO - Added KR to Win the Indy 500.: Increase average lap speed
2026-01-05 15:05:48,637 - okr_tracker - WARNING - Could not parse date 'Pre-race season', using default: 2026-04-01
2026-01-05 15:05:48,637 - okr_tracker - INFO - Initialized KR tracker: Test at wind tunnel
2026-01-05 15:05:48,638 - okr_tracker - INFO - Added KR to Win the Indy 500.: Test at wind tunnel
2026-01-05 15:05:48,638 - okr_tracker - WARNING - Could not parse date 'Race Day', using default: 2026-04-01
2026-01-05 15:05:48,639 - okr_tracker - INFO - Initialized KR tracker: Reduce average pit stop time
2026-01-05 15:05:48,639 - okr_tracker - INFO - Added KR t

📅 Timeline: Started 2025-10-01, Ends 2026-04-01
⏱️  Expected Progress: 52.7%

📊 Updating Key Results with good progress...

✅ OKR Status: On Track
📈 Actual Progress: 60.0%
🎯 Expected Progress: 52.7%
📊 Gap: -7.3% (ahead)

🔔 Health Alerts (0):
  ✨ No alerts - Everything looks great!


### Comparison Summary: All Three Statuses

In [46]:
# Compare all three trackers
import pandas as pd

comparison_data = []

for name, tracker in [
    ("At Risk", at_risk_tracker),
    ("Behind", behind_tracker),
    ("On Track", on_track_tracker)
]:
    summary = tracker.get_summary()
    comparison_data.append({
        "Scenario": name,
        "Status": summary['status'],
        "Actual %": f"{summary['progress']['actual_percent']:.1f}%",
        "Expected %": f"{summary['progress']['expected_percent']:.1f}%",
        "Gap": f"{summary['progress']['expected_percent'] - summary['progress']['actual_percent']:.1f}%",
        "Days Left": summary['period']['days_remaining'],
        "Alerts": len(summary['alerts']),
        "KR On Track": summary['key_results']['on_track'],
        "KR Behind": summary['key_results']['behind'],
        "KR At Risk": summary['key_results']['at_risk']
    })

df = pd.DataFrame(comparison_data)
print("\n📊 OKR STATUS COMPARISON")
print("="*100)
print(df.to_string(index=False))
print("="*100)

# Show health scores
print("\n🏥 HEALTH SCORES:")
for name, tracker in [("At Risk", at_risk_tracker), ("Behind", behind_tracker), ("On Track", on_track_tracker)]:
    from okr_tracker import calculate_okr_health_score
    score, desc = calculate_okr_health_score(tracker)
    print(f"  {name:12} → {score:3}/100 - {desc}")


📊 OKR STATUS COMPARISON
Scenario   Status Actual % Expected %   Gap  Days Left  Alerts  KR On Track  KR Behind  KR At Risk
 At Risk  At Risk    20.0%      52.7% 32.7%         86       1            1          4           0
  Behind   Behind    60.0%      64.6%  4.6%         86       1            3          2           0
On Track On Track    60.0%      52.7% -7.3%         86       0            3          2           0

🏥 HEALTH SCORES:
  At Risk      →  62/100 - Good - Minor attention needed
  Behind       → 90.39094650205762/100 - Excellent - On track with no major concerns
  On Track     → 100/100 - Excellent - On track with no major concerns
